# Dataset (map-style)

A dataset that returns data **by index**: `dataset[i]` gives sample `i`. It usually loads everything into RAM, but that's not required (you can open the file inside `__getitem__` instead).

### 1. Structure

```python
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, x):
        self.x = x

    def __len__(self):                # dataset size
        return len(self.x)

    def __getitem__(self, idx):       # sample by index
        return self.x[idx]

dataset = MyDataset(x=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print(dataset[0])     # calls __getitem__ -> 1
print(len(dataset))   # calls __len__ -> 10

loader = DataLoader(dataset, batch_size=4, shuffle=True)
```

### 2. Characteristics

- **Index access:** `dataset[i]` and `len(dataset)` work.
- **Easy shuffle:** the DataLoader shuffles the indices (`shuffle=True`).
- **High memory:** usually keeps everything in RAM (for large datasets, open the file in `__getitem__` or use `IterableDataset`).
- **Works well with `num_workers > 0`:** the DataLoader splits indices among workers, with no duplicate samples.

### 3. Map-style vs Iterable

| | `Dataset` | `IterableDataset` |
|---|---|---|
| Access | by index | sequential |
| `len()` | yes | no (by default) |
| Shuffle in loader | yes | no |
| Memory | high | low |
| Best for | datasets that fit in RAM | huge datasets / streaming |

In [ ]:
from torch.utils.data import Dataset


class Dataset(Dataset):
    def __init__(self):
        self.x = [[1,2,3], [4,5,6], [7,8,9]]
        self.y = [10,         20,      30]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = Dataset()


In [ ]:
print(dataset[0])

x, y = dataset[2]
print(x, y)

print(len(dataset))

([1, 2, 3], 10)
[7, 8, 9] 30
3


# IterableDataset

A dataset that yields data **sample by sample** via `__iter__` + `yield` (it's a generator). Ideal for large datasets, since it reads **shards/chunks on demand** without loading everything into memory.

### 1. Structure

```python
from pathlib import Path
from torch.utils.data import IterableDataset, DataLoader

class MyDataset(IterableDataset):
    def __init__(self, path):
        self.path = Path(path)

    def __iter__(self):
        for shard in sorted(self.path.glob("shard_*")):        # data/shard_0, data/shard_1, ...
            for chunk in sorted(shard.glob("chunk_*")):        # data/shard_0/chunk_0, ...
                data = load(chunk)                             # open the file (e.g. np.load, torch.load)
                yield data["features"], data["label"]          # yield 1 sample and pause

dataset = MyDataset("data/")
loader = DataLoader(dataset, batch_size=32)
```

> Note: `chunk` is a `Path`, not a dict. You must **load the file** before accessing `["features"]`.

### 2. Characteristics

- **Sequential access:** yields sample by sample, no index (`dataset[i]` doesn't exist).
- **Low memory:** only the current shard/chunk needs to be loaded.
- **No `shuffle=True` in the DataLoader:** there are no indices to shuffle. Alternatives: shuffle the shard order, use a shuffle buffer, or shuffle the data when creating the shards.
- **No `len()`** by default (unknown size).
- **Careful with `num_workers > 0`:** each worker runs the whole `__iter__`, producing duplicate samples unless you split the shards among them (`get_worker_info()`).

### 3. When to use

| Situation | Use |
|---|---|
| Dataset fits in RAM, easy shuffle | `Dataset` (map-style) |
| Huge dataset, sharded, streaming | `IterableDataset` |

---

In [ ]:
from torch.utils.data import IterableDataset
from pathlib import Path

path = Path('datasets/shards_train/')

class Iterable(IterableDataset):
    def __init__(self, data):
        self.data = data


    def __iter__(self):
        shards = sorted(self.data.glob('shard_*.bin')) # Organize the paths into sorted and put in a list
        # So here gonna be ['shard1.bin', 'shard2.bin',  'shard3.bin', 'shard4.bin']

        for shard in shards: # shard1.bin -> shard2.bin and more, so here each shard gonna be an PATH yet
            data_shard = abrir_shard(shard) # This option gonna open the shard, to get the values
            for sample in data_shard: # For sample inside te shard (like images)
                x = sample['features']
                y = sample['label']
       
                yield x, y # Return X and Y, but dont stop the code running 

dataset = Iterable(data = path)

# Extras

## Hugging Face Datasets (streaming)

When loading a dataset from the Hugging Face Hub, you can enable **streaming**. Instead of downloading everything, Hugging Face fetches the data **on demand**, so you just iterate with a `for` loop. Under the hood it returns an `IterableDataset`.

```python
from datasets import load_dataset

dataset = load_dataset("username/dataset_name", split="train", streaming=True)

for sample in dataset:        # fetches samples lazily, one by one
    x, y = sample["features"], sample["label"]
```

### Characteristics

- **No full download:** data is streamed from the Hub, so disk and RAM usage stay low.
- **Sequential access:** no `dataset[i]` and no `len()`, same as `IterableDataset`.
- **Shuffle with a buffer:** `dataset.shuffle(seed=42, buffer_size=10_000)` (approximate shuffle).
- **Works with PyTorch:** can be passed directly to a `DataLoader`.
- **Useful methods:** `.take(n)` to peek at the first `n` samples, `.map()` and `.filter()` (applied lazily).

```python
for sample in dataset.take(3):
    print(sample)
```

In [13]:
from torch.utils.data import IterableDataset
from pathlib import Path
from datasets import load_dataset

data = load_dataset('benjamin-paine/imagenet-1k-256x256', split = 'train', streaming = True)  # The streaming og Hugging face controlls all the thing

x = 0

print(data)


for sample in data:
    print(sample)
    x += 1


    if x == 10:
        break

IterableDataset({
    features: ['image', 'label'],
    num_shards: 40
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC84344F50>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC844042D0>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC84344F50>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC844042D0>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC84344F50>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC844042D0>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC84344F50>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC844042D0>, 'label': 0}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x1DC84344F50>, 'l

# Large Datasets 


where the data are in


1. Dataset complete = SSD/HD/S3/HF SERVER/ OTHER
2. Read of Shards (read one in the mount) = RAM
3. Chunk = RAM
4. SAMPLES = RAM
5. BATCH = RAM -> VRAM 